# EDA on log2FC values from current bulk RNA-seq dataset

## Data loading

Configure root with local/colab.

In [ ]:
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
root = Path.cwd().parent
sys.path.insert(0, str(root))

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.eda import get_annotations
from src.dge_data import (
    get_l2fc,
    get_pval,
    get_synergy
)

annotations = get_annotations(root) # Annotations
l2fc_df = get_l2fc(root) # log2fc data
pval_df = get_pval(root) # Adjusted pvalues for DEGs
synergy_df = get_synergy(root) # Interaction scores and synergy scores

## Identify DEG counts over time and dose.

Plot # DEGs over time.

In [ ]:
from src.eda import plot_degs_over_time

# Example plot for RIF
plot_degs_over_time(
    l2fc_df = l2fc_df,
    pval_df = pval_df,
    pval_cutoff = 0.05,
    l2fc_cutoff = 1,
    drug_id = "RIF"
)

## Heatmaps for log2FC data

Annotation categories.

In [ ]:
print(annotations["Category1"].unique())

Plot specific pathways and samples in heatmap of log2FC data.

In [ ]:
from src.eda import plot_l2fc_heatmap, find_consistent_interaction_genes

# Example usage to plot specific drug samples and specific pathways
drug_order = ["CIP", "CEF", "VNC", "RIF"]
annot_col = "Category1" # Annotation column
gene_categories = ["Lipid metabolism"]

plot_l2fc_heatmap(
    df = l2fc_df,
    annotations = annotations,
    secondary_annot_col = "Product",
    drug_order = drug_order,
    annot_col = annot_col,
    gene_categories = gene_categories,
    figsize = (19, 9),
    show_xticklabels = True,
    show_yticklabels = False,
    vmax = 2.5
)

Examine consistent DEGs across all samples of 1 drug.

In [ ]:
from src.eda import plot_consistent_degs

# Plot DEGs that are consistent across all CEF+CIP samples
plot_consistent_degs(
    l2fc_df = l2fc_df,
    pval_df = pval_df,
    annotations = annotations,
    drug_id = "CEF+CIP",
    pval_cutoff = 0.05,
    l2fc_cutoff = 1,
    min_fraction = 0.5, # Find DEGs that are present in at least 50% of samples
    figsize = (10, 20)
)

## Transcriptional interaction scores

Distribution of interaction scores.

In [ ]:
# Subset to combo I'm interested in
combo = "CIP+VNC"
combo_df = synergy_df[synergy_df["drug_id"] == combo]

# Get interation scores
interaction_scores = combo_df.iloc[:, combo_df.columns.str.contains("SP")].values.ravel()

# Plot histogram
fig, ax = plt.subplots()
ax.hist(interaction_scores, bins = 40)
ax.set_title(f"Distribution of interaction scores for {combo}")
ax.set_xlabel("Transcriptional interaction score")
ax.set_ylabel("Frequency")
plt.show()

Plot genes that have consistently significant interaction scores across all samples from 1 drug combination.

In [ ]:
from src.eda import find_consistent_interaction_genes

# Top x percent of interaction scores to use
top_percent = 0.05

# Get 95 and 5 quantiles
left_cutoff = np.quantile(interaction_scores, top_percent)
right_cutoff = np.quantile(interaction_scores, 1- top_percent)

print(f"{1 - top_percent} quantile : {right_cutoff}")
print(f"{top_percent} quantile : {left_cutoff}" )

# Find genes that are either consistenly above or below cutoffs in 50% of samples
consistent_interaction_genes = find_consistent_interaction_genes(
    df = synergy_df,
    combo = combo,
    left_cutoff = left_cutoff,
    right_cutoff = right_cutoff,
    min_fraction = 0.50,
)

# Heatmap 
fig, ax = plt.subplots(figsize = (15, 9))
sns.heatmap(
    data = combo_df[consistent_interaction_genes.index].T, 
    cmap = "coolwarm", 
    ax = ax, 
    vmax = 7,
    vmin = -7,
    cbar_kws = {"label": "Transcriptional interaction score"}
)

# Add annotations as extra axis
annots = annotations.reindex(consistent_interaction_genes.index)
secax = ax.secondary_yaxis("left")
secax.set_yticks(np.arange(len(annots)) + 0.5)
secax.set_yticklabels(annots["Product"])
secax.spines["left"].set_position(("outward", 80))
secax.set_ylabel("Product")

ax.set_title(f"Top consistently high/low interaction scores for {combo}")

Plot log2FC values of the genes identified above.

In [ ]:
drug_order = ["CIP", "VNC", "CIP+VNC"]
annot_col = "Category1"
genes = consistent_interaction_genes.index
plot_l2fc_heatmap(
    df = l2fc_df,
    annotations = annotations,
    drug_order = drug_order,
    annot_col = annot_col,
    figsize = (15, 9),
    show_xticklabels = True,
    show_yticklabels = True,
    genes = genes,
    vmax = 5
)

## Synergy scores

Plot synergy score distribution for each drug combination.

In [ ]:
sns.stripplot(synergy_df, x = "drug_id", y = "synergy_score", hue = "timepoint")
plt.title("Synergy score distribution")
plt.xlabel("Drug combination")
plt.ylabel("EOB synergy score")